# Evaluation, comparison, and ranking

This notebook demonstrates the SDK evaluation pipeline and the ranked-variations helper. Evaluation can call Azure AI Vision embeddings and Azure OpenAI LLM judges, so keep batches small and set `RUN_EVALUATION=1` only when you intend to incur costs.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "packages" / "t2i_core").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "packages" / "t2i_core" / "src"))
load_dotenv(PROJECT_ROOT / ".env")

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_EVALUATION = os.getenv("RUN_EVALUATION") == "1"
print("Paid generation/evaluation calls enabled:", RUN_EVALUATION)

In [ ]:
from t2i_core import EvaluationPipeline, GPTImageProvider, Settings
from t2i_core.scenarios import generate_ranked_variations

settings = Settings()
print("Evaluation weights:", settings.eval_weights)
print("Evaluation deployments:", settings.eval_model_config.model_dump())

## Generate, evaluate, and rank variations

The helper generates `n` images, evaluates each selected layer, sorts by `composite_score`, and returns ranked results.

In [ ]:
prompt = (
    "Photorealistic square hero image of a compact trail running backpack on a mossy rock, "
    "misty evergreen forest background, soft diffused morning light, premium outdoor brand style, no text."
)
layers = ["embedding", "rubric", "judge"]

if RUN_EVALUATION:
    provider = GPTImageProvider(settings)
    pipeline = EvaluationPipeline(settings)
    try:
        ranked = await generate_ranked_variations(
            provider,
            prompt,
            n=3,
            quality="high",
            pipeline=pipeline,
            layers=layers,
        )
    finally:
        await pipeline.embedding.http_client.aclose()
        await provider.client.close()

    rows = []
    for item in ranked:
        image_path = OUTPUT_DIR / f"rank-{item.rank}.png"
        report_path = OUTPUT_DIR / f"rank-{item.rank}.evaluation.json"
        image_path.write_bytes(item.generation.image)
        if item.evaluation is not None:
            report_path.write_text(item.evaluation.model_dump_json(indent=2), encoding="utf-8")
        rows.append({
            "rank": item.rank,
            "model": item.generation.model,
            "composite_score": item.evaluation.composite_score if item.evaluation else None,
            "decision": item.evaluation.threshold_decision if item.evaluation else None,
            "image_path": str(image_path),
        })
    ranking = pd.DataFrame(rows)
else:
    ranking = pd.DataFrame([
        {"rank": 1, "model": "gpt-image-2", "composite_score": 0.91, "decision": "accept", "image_path": "example-rank-1.png"},
        {"rank": 2, "model": "gpt-image-2", "composite_score": 0.84, "decision": "review", "image_path": "example-rank-2.png"},
        {"rank": 3, "model": "gpt-image-2", "composite_score": 0.72, "decision": "regenerate", "image_path": "example-rank-3.png"},
    ])
    print("Set RUN_EVALUATION=1 to generate real images and evaluation reports.")

ranking

In [ ]:
def compare_ranked_results(ranking_frame: pd.DataFrame) -> pd.DataFrame:
    """Return a compact comparison table for review or downstream automation."""

    columns = ["rank", "model", "composite_score", "decision", "image_path"]
    return ranking_frame.loc[:, columns].sort_values(["rank", "composite_score"], ascending=[True, False])


comparison = compare_ranked_results(ranking)
comparison

For production jobs, persist both the generated image and `EvaluationReport.model_dump_json(...)` so later reviewers can audit prompt, model, layer scores, threshold decisions, and cost metadata.